# 07 - Provenance and SCD2
# Enterprise BFSI Client MDM Platform

### Responsibilities:
   1. Persist attribute-level provenance
   2. Build the current mastered state
   3. Maintain SCD Type 2 history

###Setup and load survivorship audit

In [0]:
from pyspark.sql import functions as F

CATALOG = "bfsi_mdm_dev"

PROVENANCE_COLUMNS = [
    "master_client_id",
    "attribute_name",
    "attribute_value",
    "source_system",
    "source_id",
    "survivorship_reason",
]

survivorship_audit = spark.table(
    f"{CATALOG}.gold.survivorship_audit"
)

print(
    f"Survivorship audit rows: "
    f"{survivorship_audit.count():,}"
)

Survivorship audit rows: 14,047,696


###Build MASTER_CLIENT_ATTRIBUTE


The final table is:

master_client_id
attribute_name
attribute_value
winning_source_system
winning_source_id
survivorship_reason

So:

MC000001 | email

can tell us:

value = rahul@example.com
source = CORE
source_id = C12345
reason = highest-ranked populated candidate...

In [0]:
master_client_attribute = (
    survivorship_audit
    .select(
        F.col("master_client_id"),
        F.col("attribute_name"),
        F.col("attribute_value"),
        F.col("source_system").alias(
            "winning_source_system"
        ),
        F.col("source_id").alias(
            "winning_source_id"
        ),
        F.col("survivorship_reason"),
    )
)

duplicate_provenance = (
    master_client_attribute
    .groupBy(
        "master_client_id",
        "attribute_name"
    )
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_provenance.limit(1).count() == 0, (
    "Multiple provenance rows found for "
    "the same master/attribute."
)

(
    master_client_attribute
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.gold.master_client_attribute"
    )
)

print("MASTER_CLIENT_ATTRIBUTE written.")

MASTER_CLIENT_ATTRIBUTE written.


###Build the current SCD2 state

In [0]:
master_client_attribute = spark.table(
    f"{CATALOG}.gold.master_client_attribute"
)

# ------------------------------------------------------------
# SCD2 compares the mastered attribute value over time.
# Missing values become an empty string so comparison is
# explicit and deterministic.
# ------------------------------------------------------------

new_state = (
    master_client_attribute
    .select(
        "master_client_id",

        F.col("attribute_name")
        .alias("field"),

        F.coalesce(
            F.col("attribute_value"),
            F.lit("")
        ).alias("value")
    )
)

run_date = (
    spark.sql(
        "SELECT current_date() AS run_date"
    )
    .first()["run_date"]
)

print(f"SCD2 run date: {run_date}")

SCD2 run date: 2026-09-28


###Detect existing history

In [0]:
history_exists = spark.catalog.tableExists(
    f"{CATALOG}.gold.master_client_history"
)

if history_exists:

    existing_history = spark.table(
        f"{CATALOG}.gold.master_client_history"
    )

    old_current_full = (
        existing_history
        .filter(
            F.col("is_current") == True
        )
    )

    duplicate_current = (
        old_current_full
        .groupBy(
            "master_client_id",
            "field"
        )
        .count()
        .filter(F.col("count") > 1)
    )

    assert (
        duplicate_current
        .limit(1)
        .count()
        == 0
    ), (
        "Existing SCD2 history contains "
        "multiple current versions."
    )

    print("Existing SCD2 history found.")

else:

    existing_history = None
    old_current_full = None

    print(
        "No existing SCD2 history. "
        "Initial load."
    )

No existing SCD2 history. Initial load.


###Build initial or incremental SCD2 history


The four logical cases

Unchanged

old exists\
new exists\
same value\
→ keep old current version

Changed

old exists\
new exists\
different value\
→ expire old\
→ insert new current\

New

old absent\
new exists\
→ insert new current

Removed

old exists\
new absent\
→ expire old


In [0]:
if not history_exists:

    # ========================================================
    # INITIAL LOAD
    # ========================================================

    final_history = (
        new_state
        .select(
            "master_client_id",
            "field",
            "value",

            F.lit(run_date)
            .cast("date")
            .alias("effective_date"),

            F.lit(None)
            .cast("date")
            .alias("expiry_date"),

            F.lit(True)
            .alias("is_current")
        )
    )

else:

    # ========================================================
    # INCREMENTAL LOAD
    # ========================================================

    old_current = old_current_full.alias("old")
    new = new_state.alias("new")

    comparison = (
        old_current
        .join(
            new,
            [
                "master_client_id",
                "field"
            ],
            "full_outer"
        )
        .withColumn(
            "old_exists",
            F.col(
                "old.master_client_id"
            ).isNotNull()
        )
        .withColumn(
            "new_exists",
            F.col(
                "new.master_client_id"
            ).isNotNull()
        )
        .withColumn(
            "same_value",
            F.col("old.value").eqNullSafe(
                F.col("new.value")
            )
        )
    )

    # --------------------------------------------------------
    # Case 1: unchanged
    # --------------------------------------------------------

    unchanged_current = (
        comparison
        .filter(
            F.col("old_exists")
            &
            F.col("new_exists")
            &
            F.col("same_value")
        )
        .select(
            F.col(
                "old.master_client_id"
            ).alias("master_client_id"),

            F.col(
                "old.field"
            ).alias("field"),

            F.col(
                "old.value"
            ).alias("value"),

            F.col(
                "old.effective_date"
            ).alias("effective_date"),

            F.col(
                "old.expiry_date"
            ).alias("expiry_date"),

            F.col(
                "old.is_current"
            ).alias("is_current")
        )
    )

    # --------------------------------------------------------
    # Case 2: changed or removed
    #
    # Old version stops being current.
    # --------------------------------------------------------

    expired_rows = (
        comparison
        .filter(
            F.col("old_exists")
            &
            (
                ~F.col("new_exists")
                |
                ~F.col("same_value")
            )
        )
        .select(
            F.col(
                "old.master_client_id"
            ).alias("master_client_id"),

            F.col(
                "old.field"
            ).alias("field"),

            F.col(
                "old.value"
            ).alias("value"),

            F.col(
                "old.effective_date"
            ).alias("effective_date"),

            F.lit(run_date)
            .cast("date")
            .alias("expiry_date"),

            F.lit(False)
            .alias("is_current")
        )
    )

    # --------------------------------------------------------
    # Case 3: new or changed
    #
    # New current version becomes active.
    # --------------------------------------------------------

    new_or_changed_rows = (
        comparison
        .filter(
            F.col("new_exists")
            &
            (
                ~F.col("old_exists")
                |
                ~F.col("same_value")
            )
        )
        .select(
            F.col(
                "new.master_client_id"
            ).alias("master_client_id"),

            F.col(
                "new.field"
            ).alias("field"),

            F.col(
                "new.value"
            ).alias("value"),

            F.lit(run_date)
            .cast("date")
            .alias("effective_date"),

            F.lit(None)
            .cast("date")
            .alias("expiry_date"),

            F.lit(True)
            .alias("is_current")
        )
    )

    # --------------------------------------------------------
    # Existing historical versions stay untouched.
    # --------------------------------------------------------

    previously_expired = (
        existing_history
        .filter(
            F.col("is_current") == False
        )
    )

    final_history = (
        previously_expired
        .unionByName(unchanged_current)
        .unionByName(expired_rows)
        .unionByName(new_or_changed_rows)
    )

print("SCD2 state prepared.")

SCD2 state prepared.


###Write SCD2

In [0]:
(
    final_history
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.gold.master_client_history"
    )
)

print(
    "MASTER_CLIENT_HISTORY written."
)

MASTER_CLIENT_HISTORY written.


###Validate SCD2

In [0]:
history = spark.table(
    f"{CATALOG}.gold.master_client_history"
)

current_versions = (
    history
    .filter(F.col("is_current") == True)
)

duplicate_current = (
    current_versions
    .groupBy(
        "master_client_id",
        "field"
    )
    .count()
    .filter(F.col("count") > 1)
)

master_count = (
    new_state
    .select("master_client_id")
    .distinct()
    .count()
)

expected_current_rows = (
    master_count * 13
)

assert (
    duplicate_current
    .limit(1)
    .count()
    == 0
), (
    "Multiple current SCD2 versions found "
    "for the same master/field."
)

assert (
    current_versions.count()
    ==
    expected_current_rows
), (
    "Current SCD2 state does not contain "
    "exactly 13 attributes per master."
)

print(
    f"History rows : {history.count():,}"
)

print(
    f"Current rows : {current_versions.count():,}"
)

print(
    f"Expired rows : "
    f"{history.filter(~F.col('is_current')).count():,}"
)

print("SCD2 validation: PASS")

History rows : 14,047,696
Current rows : 14,047,696
Expired rows : 0
SCD2 validation: PASS


###Notebook 07 summary

In [0]:
print("=" * 70)
print("NOTEBOOK 07 COMPLETE")
print("=" * 70)

print(
    f"Provenance rows : "
    f"{master_client_attribute.count():,}"
)

print(
    f"History rows    : "
    f"{history.count():,}"
)

print(
    f"Current rows    : "
    f"{current_versions.count():,}"
)

print("MASTER_CLIENT_ATTRIBUTE: PASS")
print("MASTER_CLIENT_HISTORY: PASS")
print("=" * 70)

NOTEBOOK 07 COMPLETE
Provenance rows : 14,047,696
History rows    : 14,047,696
Current rows    : 14,047,696
MASTER_CLIENT_ATTRIBUTE: PASS
MASTER_CLIENT_HISTORY: PASS
